# einops-repeat-broadcast — worked example 3: Pairwise squared distances via double broadcast

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Composing two `einops.repeat` broadcasts gives the every-query-with-every-reference grid without ever materializing the `(N, M, D)` copies. Subtracting the two stride-zero views and reducing over the feature axis yields the full `(N, M)` distance matrix in a couple of lines — the k-nearest-neighbour and clustering primitive.

## Worked solution

Given `queries` `(N, D)` and `refs` `(M, D)`, we want an `(N, M)` matrix of squared Euclidean distances.

1. Record the counts: `N = queries.shape[0]`, `M = refs.shape[0]`.
2. Expand the queries to `(N, M, D)` with `repeat(queries, 'n d -> n m d', m=M)`. The new `m` axis is stride zero.
3. Expand the refs the opposite way to `(N, M, D)` with `repeat(refs, 'm d -> n m d', n=N)`; here the new `n` axis is stride zero.
4. Now the two grids are aligned: position `[i, j]` holds query `i` and reference `j`. Subtract them, square elementwise, and sum over the last axis `D`. That collapse turns the `(N, M, D)` difference into the `(N, M)` distance matrix.
5. The broadcast machinery supplied the pairing; the reduction supplied the contraction over feature dimensions.

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(2)
queries = t.randn(4, 3)
refs = t.randn(6, 3)

def pairwise_sq_dist(queries, refs):
    N = queries.shape[0]
    M = refs.shape[0]
    q = repeat(queries, 'n d -> n m d', m=M)
    r = repeat(refs, 'm d -> n m d', n=N)
    return ((q - r) ** 2).sum(dim=-1)

D = pairwise_sq_dist(queries, refs)
print(D.shape)
print('d[0,0] matches manual:', bool(t.isclose(D[0, 0], ((queries[0] - refs[0]) ** 2).sum())))